# 🔔 Citadel Gateway Alerting - Testing Center

## Overview

Validate the gateway's **comprehensive alerting** end-to-end by *actually triggering* two different alerts.

This notebook provisions a deliberately **restrictive access contract** and drives load through it in two
phases to trip the two `llm-token-limit` enforcement dimensions:

1. **Throttling (HTTP 429)** — a concurrent burst overshoots the per-minute **token rate (TPM)**.
2. **Quota-exceeded (HTTP 403 / `AITokenQuotaExceeded`)** — sustained load exhausts the long-term
   **token-quota**.

It then deploys **Azure Monitor alert rules** for the `AI Gateway Alert` custom metric (emitted by the
[`raise-alert-events`](../bicep/infra/modules/apim/policies/frag-raise-alert-events.xml) policy fragment)
— one rule per `alertType` — and confirms the events + alert rules fire.

| Scenario | Trigger | `alertType` |
|---|---|---|
| Ops-AlertProbe (Phase 1) | Concurrent burst exceeds the per-minute TPM → **429** | `throttling` |
| Ops-AlertProbe (Phase 2) | Sustained load exhausts the hourly token-quota → **403** | `quota-exceeded` |

> **Default alert mode = `logQuery` (log search alerts).** By default the rules are **scheduled query
> alerts** on the App Insights `customMetrics` table. This works **immediately** because the metric
> telemetry lands in the logs the moment the gateway emits it. Metric alerts on the pre-aggregated
> `ai-gateway-alerts` namespace are also supported (set `alert_mode = "metric"`), but that namespace
> only **registers after the metric has first been emitted** — so it can't be created up front and isn't
> visible under App Insights → Metrics/Alerts until then. That's why log-query is the default.

## Expected outcomes

- A restrictive access contract (APIM product + subscription) deployed via Bicep with a very low
  `tokens-per-minute` **and** a small `token-quota`.
- A **model auto-selected** from the gateway's supported models (a `gpt`-family model, excluding
  image / embedding / non-Azure cloud models), with an override option.
- **Phase 1**: a burst that produces a wave of **HTTP 429** (`throttling`).
- **Phase 2**: sustained load that exhausts the quota and produces **HTTP 403** (`quota-exceeded`).
- Azure Monitor **alert rules** (`throttling` + `quota-exceeded`, log-query by default) and an
  **email Action Group** deployed via [`bicep/infra/app-insights-alert`](../bicep/infra/app-insights-alert/README.md).
- Verification that the `AI Gateway Alert` events were emitted for **both** alertTypes and the rules exist.

## Azure Prerequisites

- An existing Citadel Governance Hub deployment (APIM + Application Insights).
- Azure credentials with permission to deploy at subscription scope and create alert rules / action
  groups in the hub resource group.
- The Azure CLI `application-insights` extension for the log verification step
  (`az extension add -n application-insights`; the notebook prompts if it's missing).
- A reachable **email address** to receive the alert notifications.

> **Note:** "Triggering" an alert is not instantaneous — this notebook emits the events and configures
> the rules; the emails arrive once Azure Monitor next evaluates the crossed thresholds (log-query rules
> evaluate on a schedule, minimum `PT1M`). The `customMetrics` telemetry itself is visible in App
> Insights **Logs** within ~1–2 minutes. See
> [Throttling & Critical Event Alerting](../guides/throttling-events-handling.md).


<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group`, `location`, and the APIM Application Insights name
  from your active `azd` environment. Works when the accelerator was deployed with `azd up`.
- `False` — fill the `REPLACE` values manually below.

Also set the **alert email** and (optionally) the **model override** and **throttling thresholds**.


In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
init_from_azd = True   # Set False to fill the REPLACE values below manually.

# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION (REQUIRED — used as defaults if azd lookup fails)
# ============================================================================
governance_hub_resource_group = "REPLACE"   # Resource group of the deployed Citadel Governance Hub
location = "REPLACE"                         # Azure region (e.g. "swedencentral", "eastus")

# ============================================================================
# 🔧 API VERSION CONFIGURATION
# ============================================================================
inference_api_version = "2024-05-01-preview"
openai_api_version    = "2024-12-01-preview"
targetInferenceApi    = "models"             # 'models' = Universal LLM API | 'openai' = Azure OpenAI API

# ============================================================================
# 🧠 MODEL SELECTION
# ----------------------------------------------------------------------------
# By default the notebook auto-selects a chat model from the gateway's supported
# models: it prefers a name containing `gpt` while EXCLUDING image, embedding,
# audio and non-Azure cloud model families. Set `model_name_override` to pin a
# specific model and skip auto-selection.
# ============================================================================
model_name_override      = ""          # e.g. "gpt-4.1" — leave empty to auto-select
gpt_include_keyword      = "gpt"
model_exclusion_keywords = ["image", "embed", "embedding", "tts", "whisper",
                             "audio", "realtime", "dall", "vision", "sora", "transcribe"]
# Non-Azure-OpenAI ("other cloud") model families to exclude even if they contain 'gpt'
cloud_exclusion_keywords = ["bedrock", "gemini", "claude", "anthropic", "mistral",
                             "llama", "titan", "nova", "cohere", "deepseek", "grok", "phi"]

# ============================================================================
# 🔐 KEY VAULT CONFIGURATION (not used by this test)
# ============================================================================
use_keyvault_integration = False

# ============================================================================
# 🤖 MICROSOFT FOUNDRY CONFIGURATION (not used by this test)
# ============================================================================
use_foundry_integration = False

# ============================================================================
# 🔔 ALERTING CONFIGURATION
# ----------------------------------------------------------------------------
# This notebook triggers TWO alert categories from the same restrictive contract:
#   • throttling      (HTTP 429) — a concurrent BURST overshoots the per-minute token rate (TPM)
#   • quota-exceeded  (HTTP 403) — sustained load exhausts the long-term token-quota
#
# alert_email_address : where the alert notification is sent (REQUIRED)
# app_insights_name   : APIM Application Insights component. Empty = auto-discover from the
#                       hub resource group (matches a name containing 'apim', else the first).
# ============================================================================
alert_email_address = "REPLACE@example.com"
app_insights_name   = ""            # empty = auto-discover in the hub resource group

# Restrictive capacity for the probe contract. Tuned so a SINGLE request succeeds (consuming
# quota) while a concurrent BURST overshoots the per-minute rate:
#   - tokens-per-minute (TPM) low  -> a 12-way burst returns HTTP 429 (throttling)
#   - hourly token-quota small     -> cumulative tokens exhaust it -> HTTP 403 (quota-exceeded)
probe_tokens_per_minute  = 500
probe_token_quota        = 3500
probe_token_quota_period = "Hourly"

# Phase 2 (quota exhaustion): keep sending until a 403 appears or the time budget elapses.
quota_max_seconds    = 360     # wall-clock budget to reach quota exhaustion (403)
quota_rate_wait_secs = 15      # fallback wait after a 429 when no Retry-After header is present

# Metric identity (must match the raise-alert-events fragment)
alert_metric_namespace = "ai-gateway-alerts"
alert_metric_name      = "AI Gateway Alert"

# --- ALERTING MODE ----------------------------------------------------------
# "logQuery" (DEFAULT): scheduled query (log search) alerts on the App Insights `customMetrics`
#            table. Works immediately because the metric telemetry lands in the logs as soon as
#            the gateway emits it — no waiting for the pre-aggregated metric namespace.
# "metric":  metric alerts on the `ai-gateway-alerts` namespace. Only works AFTER the custom
#            metric has been emitted at least once and the namespace has registered (can take a
#            few minutes, and until then it is NOT visible under App Insights → Metrics/Alerts).
alert_mode = "logQuery"

# Demo-tuned alert rules (fire fast; raise the window/threshold for production)
throttling_alert_threshold = 3       # > 3 throttling (429) events...
throttling_alert_window    = "PT1M"  # ...within 1 minute
quota_alert_threshold      = 0       # ANY quota-exceeded (403) event fires the alert
quota_alert_window         = "PT1M"
alert_evaluation_freq      = "PT1M"
alert_severity             = 2       # Warning

def _is_unset(value):
    return value is None or value == "" or value == "REPLACE"

if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group":       ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":             ["AZURE_LOCATION", "LOCATION"],
        "subscription_id":      ["AZURE_SUBSCRIPTION_ID"],
        "apim_app_insights":    ["APIM_APP_INSIGHTS_NAME"],
    }, verbose=False)

    if _is_unset(governance_hub_resource_group) and "resource_group" in loaded:
        governance_hub_resource_group = loaded["resource_group"]
    if _is_unset(location) and "location" in loaded:
        location = loaded["location"]
    if _is_unset(app_insights_name) and "apim_app_insights" in loaded:
        app_insights_name = loaded["apim_app_insights"]

    utils.print_ok(f"Resource group : {governance_hub_resource_group}")
    utils.print_ok(f"Location       : {location}")
    utils.print_ok(f"App Insights   : {app_insights_name or '(auto-discover)'}")

if _is_unset(alert_email_address) or alert_email_address == "REPLACE@example.com":
    utils.print_warning("Set 'alert_email_address' to a real inbox to receive the alert notification.")

utils.print_ok(f"Notebook variables initialized! (alert_mode = {alert_mode})")


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:


In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")


<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool & Auto-Select Model

Initialize the APIM client against your Governance Hub, read the supported models from the
`set-backend-pools` policy fragment, and auto-select a `gpt`-family chat model (excluding image /
embedding / audio / non-Azure cloud models). Set `model_name_override` in the init cell to bypass this.


In [ ]:
def select_default_model(models):
    """Pick a chat model for the load test.
       Preference: name contains `gpt` AND does not contain any image / embedding /
       audio / non-Azure-cloud exclusion keyword. Honors `model_name_override`."""
    if model_name_override:
        return model_name_override
    candidates = []
    for m in models:
        ml = m.lower()
        if gpt_include_keyword not in ml:
            continue
        if any(x in ml for x in model_exclusion_keywords):
            continue
        if any(x in ml for x in cloud_exclusion_keywords):
            continue
        candidates.append(m)
    if candidates:
        return candidates[0]
    # Fallbacks: any gpt-ish model, else first supported, else a sensible default
    gpt_any = [m for m in models if gpt_include_keyword in m.lower()]
    if gpt_any:
        return gpt_any[0]
    return models[0] if models else "gpt-4.1"

try:
    apimClientTool = APIMClientTool(governance_hub_resource_group)
    apimClientTool.initialize()
    apimClientTool.discover_api(targetInferenceApi)

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    azure_endpoint = str(apimClientTool.azure_endpoint)

    # Discover supported models from the policy fragment
    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Supported models: {supported_models}")

    # Auto-select (or honor override)
    model_name = select_default_model(supported_models)
    if model_name_override:
        utils.print_ok(f"Using model (override): {model_name}")
    else:
        utils.print_ok(f"Auto-selected model: {model_name}")

    if targetInferenceApi == "openai":
        chat_completions_url = f"{azure_endpoint}openai/deployments/{model_name}/chat/completions?api-version={inference_api_version}"
    else:  # models
        chat_completions_url = f"{azure_endpoint}models/chat/completions?api-version={inference_api_version}"
    utils.print_info(f"Chat Completion Endpoint: {chat_completions_url}")
    utils.print_info(f"Using API: {apimClientTool.api_id}")

    utils.print_ok("Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")


<a id='3'></a>
### 3️⃣ Define the Restrictive Throttling Access Contract

Define a single **Ops-AlertProbe** access contract with a deliberately low `tokens-per-minute` and
`token-quota`, so a short burst of requests exceeds capacity and returns **HTTP 429**. The generated
policy also explicitly opts the contract into `alertOnThrottling` and `alertOnAuthFailure` alerting
(throttling is on by default; this makes the intent visible for the demo).


In [ ]:
timestamp = time.strftime('%Y%m%d%H%M%S')

access_contract = {
    "name": f"ops-alertprobe-contract-{timestamp}",
    "business_unit": "Ops",
    "use_case_name": "AlertProbe",
    "environment": "DEV",
    "use_keyvault": False,
    "use_foundry": False,
    "endpoint_secret": "OPS-ALERTPROBE-LLM-ENDPOINT",
    "apikey_secret": "OPS-ALERTPROBE-LLM-KEY",
    "description": "Ops Alert Probe - restrictive TPM to trigger throttling alerts",
    "allowed_models": [model_name],
    "tokens_per_minute": probe_tokens_per_minute,
    "token_quota": probe_token_quota,
    "token_quota_period": probe_token_quota_period,
}

# Product ID (hyphenated) — used by APIM as the product-id and for subscription lookup / deletion.
product_id = f"LLM-{access_contract['business_unit']}-{access_contract['use_case_name']}-{access_contract['environment']}"

# Product DISPLAY NAME (space-separated) — this is what `context.Product.Name` returns and therefore
# what the gateway emits as the `productName` custom dimension. Alert filters must use THIS value,
# NOT the hyphenated product_id. Matches the access-contract Bicep: '<code> <BU> <UseCase> <Env>'.
product_display_name = product_id.replace("-", " ")

utils.print_info("Defined restrictive throttling access contract:")
utils.print_info(f"  Product ID     : {product_id}")
utils.print_info(f"  Product Name   : {product_display_name}  (emitted as the 'productName' dimension)")
utils.print_info(f"  Allowed models : {', '.join(access_contract['allowed_models'])}")
utils.print_info(f"  Capacity       : {access_contract['tokens_per_minute']} TPM, quota {access_contract['token_quota']} / {access_contract['token_quota_period']}")


<a id='3.1'></a>
### 3️⃣.1 Generate the Access Contract Parameter File & Dynamic Policy

Generate the Bicep parameter file and a **dynamically-generated APIM product policy** that applies:
model RBAC, the restrictive `llm-token-limit`, and explicit alerting opt-ins
(`alertOnThrottling` / `alertOnAuthFailure`).


In [ ]:
bicep_dir = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

def build_product_policy_xml(contract):
    """Per-contract APIM product policy: model RBAC + restrictive capacity + explicit alerting opt-ins.
       Alerting toggles are set in <inbound> (before downstream fragments) so they are in scope when
       raise-alert-events runs in <outbound>/<on-error>."""
    allowed_csv = ",".join(contract["allowed_models"])
    return f'''<policies>
    <inbound>
        <base />
        <!-- Alerting opt-ins (throttling is on by default; set explicitly for the demo) -->
        <set-variable name="alertsEnabled" value="true" />
        <set-variable name="alertOnThrottling" value="true" />
        <set-variable name="alertOnAuthFailure" value="true" />

        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Model-level RBAC: only the models below are allowed for this contract -->
        <set-variable name="allowedModels" value="{allowed_csv}" />
        <include-fragment fragment-id="validate-model-access" />

        <!-- Restrictive capacity allocation: forces 429 throttling under load -->
        <llm-token-limit counter-key="@(context.Subscription.Id)"
                         tokens-per-minute="{contract["tokens_per_minute"]}"
                         estimate-prompt-tokens="true"
                         token-quota="{contract["token_quota"]}"
                         token-quota-period="{contract["token_quota_period"]}"
                         retry-after-header-name="retry-after"
                         remaining-tokens-header-name="remaining-tokens" />

        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend><base /></backend>
    <outbound><base /></outbound>
    <on-error><base /></on-error>
</policies>'''

# Folder structure: contracts/[businessunit-usecase]/[environment]/
folder_name = f"{access_contract['business_unit'].lower()}-{access_contract['use_case_name'].lower()}"
contract_folder = os.path.join(bicep_dir, "contracts", folder_name, access_contract['environment'].lower())
os.makedirs(contract_folder, exist_ok=True)

policy_path = os.path.join(contract_folder, "ai-product-policy.xml")
with open(policy_path, "w", encoding="utf-8") as f:
    f.write(build_product_policy_xml(access_contract))
utils.print_ok(f"📋 Generated dynamic policy: {policy_path}")

params_file = os.path.join(contract_folder, "main.bicepparam")
params_content = f"""using '../../../main.bicep'

// Ops Alert Probe - restrictive TPM contract generated from citadel-alerting-tests.ipynb
param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: 'placeholder'
}}

param useTargetAzureKeyVault = false

param useCase = {{
  businessUnit: '{access_contract['business_unit']}'
  useCaseName: '{access_contract['use_case_name']}'
  environment: '{access_contract['environment']}'
}}

param apiNameMapping = {{
  LLM: ['universal-llm-api', 'azure-openai-api', 'unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: '{access_contract['endpoint_secret']}'
    apiKeySecretName: '{access_contract['apikey_secret']}'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Access Contract created from citadel-alerting-tests.ipynb - {access_contract["description"]}'

param useTargetFoundry = false
param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
"""
with open(params_file, "w", encoding="utf-8") as f:
    f.write(params_content)
utils.print_ok(f"✅ Parameter file created: {params_file}")


<a id='3.2'></a>
### 3️⃣.2 Deploy the Access Contract & Retrieve its Key

Deploy the contract with 🦾 Bicep, re-initialize the APIM client to pick up the new subscription, and
retrieve the subscription key used to call the gateway.


In [ ]:
deployment_cmd = (
    f"az deployment sub create --name {access_contract['name']} "
    f"--location {location} --template-file {template_file} --parameters {params_file}"
)
output = utils.run(
    deployment_cmd,
    f"Deployment '{access_contract['name']}' succeeded",
    f"Deployment '{access_contract['name']}' failed",
)
contract_deployed = output.success

# Re-initialize APIM client to pick up the new subscription
apimClientTool.initialize()

# Retrieve the subscription key for the probe contract
api_key = None
subscription_name = f"{product_id}-SUB-01"
for sub in apimClientTool.apim_subscriptions:
    if subscription_name.lower() in sub.get('name', '').lower():
        api_key = sub.get('key')
        break

if api_key:
    utils.print_ok(f"🔑 Retrieved key for {product_id}")
else:
    utils.print_error(f"Could not find subscription key for {product_id}. Check the deployment output above.")


<a id='4'></a>
### 4️⃣ Discover Application Insights & Deploy Azure Monitor Alerts

Resolve the APIM Application Insights component (auto-discovered from the hub resource group unless
`app_insights_name` was set), then deploy the alert rules + email action group from
[`bicep/infra/app-insights-alert`](../bicep/infra/app-insights-alert/README.md).


In [ ]:
# Auto-discover the APIM Application Insights component if not explicitly set.
if _is_unset(app_insights_name):
    utils.print_info("Discovering Application Insights components in the hub resource group...")
    ai_list = utils.run(
        f"az resource list -g {governance_hub_resource_group} "
        f"--resource-type Microsoft.Insights/components --query \"[].name\" -o json",
        "Listed Application Insights components",
        "Failed to list Application Insights components",
    )
    names = ai_list.json_data if (ai_list.success and ai_list.json_data) else []
    if names:
        # Prefer the APIM component (name contains 'apim'), else take the first.
        apim_ai = next((n for n in names if "apim" in n.lower()), names[0])
        app_insights_name = apim_ai
        utils.print_ok(f"Using Application Insights: {app_insights_name}")
        if len(names) > 1:
            utils.print_info(f"   (candidates: {names})")
    else:
        utils.print_error("No Application Insights component found. Set 'app_insights_name' manually.")
else:
    utils.print_ok(f"Using Application Insights (explicit): {app_insights_name}")


<a id='4.1'></a>
### 4️⃣.1 Deploy the Alert Rules & Email Action Group

Generate a Bicep parameter file scoped to this probe contract (`productNameFilter = product_id`) with
demo-tuned **throttling** (429) and **quota-exceeded** (403) rules, then deploy
`app-insights-alert/main.bicep` at subscription scope.

> **Default mode = `logQuery`.** The rules are **scheduled query (log search) alerts** on the App
> Insights `customMetrics` table, which works **immediately** — the metric telemetry lands in the logs
> as soon as the gateway emits it. Metric alerts on the `ai-gateway-alerts` namespace are also
> supported (set `alert_mode = "metric"` in the init cell), but that namespace only registers **after**
> the metric has first been emitted, so it can't be created up front and isn't visible under
> App Insights → Metrics / Alerts until then.


In [ ]:
alert_bicep_dir = "../bicep/infra/app-insights-alert"
alert_template = os.path.join(alert_bicep_dir, "main.bicep")
alert_params_file = os.path.join(alert_bicep_dir, f"generated.{product_id.lower()}-local.bicepparam")

alert_params_content = f"""using 'main.bicep'

// Generated by citadel-alerting-tests.ipynb — scoped to a single probe contract.
param appInsights = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{app_insights_name}'
}}

param alertEmailAddress = '{alert_email_address}'
// productNameFilter must match the 'productName' custom dimension, which is the APIM product
// DISPLAY NAME (space-separated, e.g. 'LLM Ops AlertProbe DEV') — NOT the hyphenated product_id.
param productNameFilter = '{product_display_name}'
param namePrefix = 'ai-gateway-alertprobe'
param actionGroupShortName = 'aigwprobe'
param metricNamespace = '{alert_metric_namespace}'
param metricName = '{alert_metric_name}'
param alertsEnabled = true

// DEFAULT 'logQuery' — scheduled query (log search) alerts on the customMetrics table (work
// immediately). Set alert_mode='metric' in the init cell once the ai-gateway-alerts metric
// namespace has registered under App Insights → Metrics.
param alertMode = '{alert_mode}'

param alerts = [
  {{
    name: 'throttling'
    alertType: 'throttling'
    severity: {alert_severity}
    operator: 'GreaterThan'
    threshold: {throttling_alert_threshold}
    timeAggregation: 'Total'
    windowSize: '{throttling_alert_window}'
    evaluationFrequency: '{alert_evaluation_freq}'
    description: 'Ops Alert Probe validation (throttling): the gateway returned HTTP 429 more than {throttling_alert_threshold} times within {throttling_alert_window} for product {product_display_name} - the per-minute token rate (TPM) was exceeded. Investigate the productName / deploymentName / backendId dimensions of the AI Gateway Alert metric. See guides/throttling-events-handling.md.'
  }}
  {{
    name: 'quota-exceeded'
    alertType: 'quota-exceeded'
    severity: {alert_severity}
    operator: 'GreaterThan'
    threshold: {quota_alert_threshold}
    timeAggregation: 'Total'
    windowSize: '{quota_alert_window}'
    evaluationFrequency: '{alert_evaluation_freq}'
    description: 'Ops Alert Probe validation (quota): the gateway returned HTTP 403 (AITokenQuotaExceeded) for product {product_display_name} - the long-term token-quota was exhausted and further calls are blocked until the window resets. Investigate the productName / deploymentName dimensions of the AI Gateway Alert metric. See guides/throttling-events-handling.md.'
  }}
]
"""

# NOTE: write as UTF-8 — the content contains non-ASCII characters (e.g. em dash / arrow) that the
# Windows default cp1252 codec cannot encode.
with open(alert_params_file, "w", encoding="utf-8") as f:
    f.write(alert_params_content)
utils.print_ok(f"✅ Alert parameter file created: {alert_params_file}")
utils.print_info(f"   Mode: {alert_mode} | Rules: throttling (429) + quota-exceeded (403), filtered by productName='{product_display_name}'.")
if alert_mode == "logQuery":
    utils.print_info("   (logQuery = scheduled query alerts on customMetrics — work immediately, no metric-namespace wait)")

alert_deployment_name = f"ai-gateway-alerts-{timestamp}"
alert_deploy_cmd = (
    f"az deployment sub create --name {alert_deployment_name} "
    f"--location {location} --template-file {alert_template} --parameters {alert_params_file}"
)
alert_output = utils.run(
    alert_deploy_cmd,
    f"Alert deployment '{alert_deployment_name}' succeeded",
    f"Alert deployment '{alert_deployment_name}' failed",
)
alerts_deployed = alert_output.success
if alerts_deployed:
    utils.print_ok("🔔 Throttling + quota-exceeded alert rules + email action group deployed. You will be emailed when a threshold is next crossed.")


<a id='5'></a>
### 5️⃣ Phase 1 — Trigger Throttling (HTTP 429)

Send a concurrent burst of chat completions through the restrictive contract. Because its
`tokens-per-minute` (TPM) is very low, the burst overshoots the per-minute rate and the gateway returns
**HTTP 429**. The `raise-alert-events` fragment emits the `throttling` alert metric on each 429.

> This is the **rate limit** (per-minute). The **quota** (long-term budget) is exhausted separately in
> Phase 2 below, which produces **HTTP 403** and the distinct `quota-exceeded` alert.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Burst configuration — enough concurrent load to exceed the low TPM and produce 429s.
burst_total_requests = 60
burst_concurrency    = 12

messages = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a detailed 200-word paragraph about cloud governance."},
    ],
}

def send_one(i):
    """Returns (status_code, total_tokens) — tokens only present on 200 responses."""
    try:
        r = requests.post(chat_completions_url, headers={"api-key": api_key}, json=messages, timeout=60)
        toks = 0
        if r.status_code == 200:
            try:
                toks = r.json().get("usage", {}).get("total_tokens", 0)
            except Exception:
                toks = 0
        return r.status_code, toks
    except Exception:
        return 0, 0

status_counts = {}
load_runs = []
tokens_consumed = 0   # cumulative — carried into Phase 2 to show quota burn-down

if not api_key:
    utils.print_error("No API key available — deploy the contract first (step 3.2).")
else:
    utils.print_info(f"Bursting {burst_total_requests} requests @ concurrency {burst_concurrency} through {product_id} ...")
    start = time.time()
    with ThreadPoolExecutor(max_workers=burst_concurrency) as ex:
        futures = [ex.submit(send_one, i) for i in range(burst_total_requests)]
        for fut in as_completed(futures):
            code, toks = fut.result()
            load_runs.append(code)
            tokens_consumed += toks
            status_counts[code] = status_counts.get(code, 0) + 1
    elapsed = time.time() - start

    throttled = status_counts.get(429, 0)
    success = status_counts.get(200, 0)
    quota_403 = status_counts.get(403, 0)
    utils.print_info(f"Completed {len(load_runs)} requests in {elapsed:.1f}s")
    utils.print_ok(f"   ✅ 200 OK        : {success}  (~{tokens_consumed} tokens toward the {probe_token_quota}/{probe_token_quota_period} quota)")
    if throttled:
        utils.print_warning(f"   ⛔ 429 Throttled : {throttled}  (throttling alert metric emitted for each)")
    else:
        utils.print_warning("   No 429s yet — lower probe_tokens_per_minute or increase the burst, then re-run.")
    if quota_403:
        utils.print_warning(f"   🚫 403 Quota     : {quota_403}  (quota already exhausting — see Phase 2)")
    other = {k: v for k, v in status_counts.items() if k not in (200, 429, 403)}
    if other:
        utils.print_info(f"   Other statuses  : {other}")


<a id='5.1'></a>
### 5️⃣.1 Phase 2 — Exhaust the Token Quota (HTTP 403)

Now keep sending **sustained** load so cumulative token usage exhausts the contract's long-term
`token-quota`. Once the quota is spent, `llm-token-limit` returns **HTTP 403 (AITokenQuotaExceeded)**
for every further call until the quota window resets — and `raise-alert-events` emits the distinct
`quota-exceeded` alert metric.

The loop paces itself around the per-minute rate limit (honoring `Retry-After` on 429s) and stops as
soon as the first 403 appears, or when `quota_max_seconds` elapses.

> **Narrative:** *first run* → 429 (the per-minute **rate** limit, Phase 1); *further runs* → 403 (the
> long-term **quota**, this phase). Two different limits, two different alerts.


In [ ]:
def send_quota_probe():
    """Single request. Returns (status_code, total_tokens, retry_after_seconds)."""
    try:
        r = requests.post(chat_completions_url, headers={"api-key": api_key}, json=messages, timeout=60)
        toks = 0
        if r.status_code == 200:
            try:
                toks = r.json().get("usage", {}).get("total_tokens", 0)
            except Exception:
                toks = 0
        ra_hdr = r.headers.get("retry-after") or r.headers.get("Retry-After") or "0"
        try:
            ra = int(float(ra_hdr))
        except Exception:
            ra = 0
        return r.status_code, toks, ra
    except Exception:
        return 0, 0, 0

quota_runs = []
quota_403 = 0

if not api_key:
    utils.print_error("No API key available — deploy the contract first (step 3.2).")
else:
    utils.print_info(
        f"Sending sustained load until the {probe_token_quota}/{probe_token_quota_period} quota is "
        f"exhausted (403) or {quota_max_seconds}s elapse. (~{tokens_consumed} tokens already used in Phase 1.)"
    )
    deadline = time.time() + quota_max_seconds
    while time.time() < deadline:
        code, toks, ra = send_quota_probe()
        quota_runs.append(code)
        tokens_consumed += toks
        if code == 403:
            quota_403 += 1
            utils.print_warning(
                f"   🚫 403 AITokenQuotaExceeded — quota exhausted (~{tokens_consumed} tokens consumed). "
                f"'quota-exceeded' alert metric emitted."
            )
            break
        elif code == 429:
            wait = ra if ra > 0 else quota_rate_wait_secs
            utils.print_info(f"   ⛔ 429 (per-minute rate) — waiting {min(wait, 30)}s for the TPM window to refill...")
            time.sleep(min(wait, 30))
        elif code == 200:
            utils.print_ok(f"   ✅ 200 — cumulative ~{tokens_consumed} / {probe_token_quota} tokens toward quota")
            time.sleep(1)
        else:
            utils.print_warning(f"   Unexpected status {code}")
            time.sleep(2)

    if quota_403 == 0:
        utils.print_warning(
            f"   No 403 within {quota_max_seconds}s. Lower probe_token_quota (init cell) or increase "
            f"quota_max_seconds, then re-run this cell."
        )


<a id='6'></a>
### 6️⃣ Verify the Alert Events & Rules

Confirm the alert events landed in the App Insights **`customMetrics` logs** (split by `alertType`,
expecting `throttling` and `quota-exceeded`) — this is the source both alert modes rely on and is
available immediately. Then confirm both alert rules exist (scheduled query rules for the default
`logQuery` mode, or metric alerts for `metric` mode).

> In `metric` mode the cell also checks whether the pre-aggregated `ai-gateway-alerts` namespace has
> registered yet (it can take a few minutes; the default `logQuery` mode does not need it).


In [ ]:
app_insights_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{governance_hub_resource_group}"
    f"/providers/microsoft.insights/components/{app_insights_name}"
)

# --- 1) Verify the alert events landed in the customMetrics LOGS (available immediately) ---
# This is the source both alert modes rely on. The pre-aggregated metric namespace (used by
# 'metric' alerts) can take minutes to register and is NOT required for logQuery alerts.
alert_totals = {}
kql = (
    "customMetrics "
    f"| where name == '{alert_metric_name}' "
    "| where timestamp > ago(30m) "
    "| summarize AggregatedValue = sum(valueSum) by alertType = tostring(customDimensions.alertType)"
)
log_out = utils.run(
    f"az monitor app-insights query --app {app_insights_name} -g {governance_hub_resource_group} "
    f"--analytics-query \"{kql}\" -o json",
    "", "",
    print_command_to_run=False,
)
if log_out.success and log_out.json_data:
    try:
        for tbl in log_out.json_data.get("tables", []):
            cols = [c["name"] for c in tbl.get("columns", [])]
            for row in tbl.get("rows", []):
                rec = dict(zip(cols, row))
                atype = rec.get("alertType") or "unknown"
                val = rec.get("AggregatedValue") or 0
                if val:
                    alert_totals[atype] = alert_totals.get(atype, 0) + val
    except Exception as e:
        utils.print_warning(f"Could not parse log query result: {e}")
elif not log_out.success:
    utils.print_warning(
        "Could not run the customMetrics log query. If the CLI reports a missing extension, run: "
        "az extension add -n application-insights. (You can also confirm in the portal: App Insights → Logs.)"
    )

metric_found = bool(alert_totals)
if metric_found:
    utils.print_ok(f"📈 customMetrics logs for '{alert_metric_name}' by alertType: {json.dumps(alert_totals)}")
    utils.print_info(f"   throttling (429)     : {int(alert_totals.get('throttling', 0))}")
    utils.print_info(f"   quota-exceeded (403) : {int(alert_totals.get('quota-exceeded', 0))}")
else:
    # Fall back to what the load phases produced (proves emission even if logs lag a little)
    ev_429 = (status_counts.get(429, 0) if 'status_counts' in dir() else 0)
    ev_403 = (quota_runs.count(403) if 'quota_runs' in dir() and quota_runs else 0)
    utils.print_info(f"Logs not visible yet; load produced 429×{ev_429}, 403×{ev_403}. customMetrics can lag ~1-2 min — re-run shortly.")

# --- 2) (metric mode only) check whether the pre-aggregated metric namespace has registered ---
if alert_mode == "metric":
    ns_out = utils.run(
        f"az monitor metrics list-namespaces --resource \"{app_insights_id}\" "
        f"--query \"[?contains(name,'ai-gateway')].name\" -o json",
        "", "", print_command_to_run=False,
    )
    if ns_out.success and ns_out.json_data:
        utils.print_ok(f"📊 Metric namespace registered: {ns_out.json_data}")
    else:
        utils.print_warning(
            "Metric namespace 'ai-gateway-alerts' not registered yet — 'metric' alerts won't evaluate until "
            "it appears (can take a few minutes after first emission). The default 'logQuery' mode avoids this wait."
        )

# --- 3) Confirm both alert rules exist (using the CLI matching the alert mode) ---
rule_names = ["ai-gateway-alertprobe-throttling-alert", "ai-gateway-alertprobe-quota-exceeded-alert"]
for rule_name in rule_names:
    if alert_mode == "logQuery":
        show_cmd = (
            f"az monitor scheduled-query show -g {governance_hub_resource_group} -n {rule_name} "
            f"--query \"{{name:name, enabled:enabled, severity:severity, windowSize:windowSize}}\" -o json"
        )
    else:
        show_cmd = (
            f"az monitor metrics alert show -g {governance_hub_resource_group} -n {rule_name} "
            f"--query \"{{name:name, enabled:enabled, severity:severity, windowSize:windowSize}}\" -o json"
        )
    rule_out = utils.run(show_cmd, "", "", print_command_to_run=False)
    if rule_out.success and rule_out.json_data:
        utils.print_ok(f"🔔 {alert_mode} alert rule: {json.dumps(rule_out.json_data)}")
    else:
        utils.print_warning(f"Alert rule '{rule_name}' not found for mode '{alert_mode}' (check step 4.1).")

utils.print_info("When Azure Monitor next evaluates a crossed threshold, the configured email will be sent.")


---
## 📊 Results Summary
---


In [ ]:
utils.print_info(f"\n{'='*72}")
utils.print_info("📊 GATEWAY ALERTING TEST SUMMARY")
utils.print_info(f"{'='*72}")

utils.print_info(f"Access contract     : {product_id}  ({'deployed' if contract_deployed else 'NOT deployed'})")
utils.print_info(f"Model used          : {model_name}")
utils.print_info(f"Capacity            : {access_contract['tokens_per_minute']} TPM, quota {access_contract['token_quota']} / {access_contract['token_quota_period']}")

# Phase 1 — throttling (429)
if load_runs:
    utils.print_info(f"\nPhase 1 — throttling (429):")
    utils.print_info(f"   Requests         : {len(load_runs)}")
    utils.print_ok(f"   200 OK           : {status_counts.get(200, 0)}")
    utils.print_info(f"   429 Throttled    : {status_counts.get(429, 0)}")

# Phase 2 — quota-exceeded (403)
if 'quota_runs' in dir() and quota_runs:
    q200 = quota_runs.count(200)
    q429 = quota_runs.count(429)
    q403 = quota_runs.count(403)
    utils.print_info(f"\nPhase 2 — quota-exceeded (403):")
    utils.print_info(f"   Requests         : {len(quota_runs)}  (~{tokens_consumed} tokens consumed total)")
    utils.print_ok(f"   200 OK           : {q200}")
    utils.print_info(f"   429 Throttled    : {q429}")
    utils.print_info(f"   403 Quota        : {q403}")

utils.print_info(f"\nApp Insights        : {app_insights_name}")
utils.print_info(f"Alert mode          : {alert_mode}  ({'scheduled query / log search' if alert_mode == 'logQuery' else 'metric alert'})")
utils.print_info(f"Alert rules deployed: {'yes (throttling + quota-exceeded)' if alerts_deployed else 'no'}")
if 'alert_totals' in dir() and alert_totals:
    utils.print_info(f"Alert events (logs) : {json.dumps(alert_totals)}")
else:
    utils.print_info("Alert events (logs) : not visible yet (retry step 6)")

throttled_ok = bool(load_runs) and status_counts.get(429, 0) > 0
quota_ok = 'quota_runs' in dir() and quota_runs.count(403) > 0
if contract_deployed and alerts_deployed and throttled_ok and quota_ok:
    utils.print_ok("\n✅ Both alerts triggered: 429 throttling (Phase 1) and 403 quota-exceeded (Phase 2). "
                   "Check the alert emails once Azure Monitor evaluates the thresholds.")
elif contract_deployed and alerts_deployed and throttled_ok:
    utils.print_warning("\n⚠️ Throttling (429) triggered, but no 403 quota event yet — re-run Phase 2 "
                        "(step 5.1) or lower probe_token_quota.")
else:
    utils.print_warning("\n⚠️ One or more steps incomplete — review the cells above.")


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the resources created by this notebook: the alert rules, the email action group, the probe
access contract (APIM product + subscription), and the generated parameter files.

> Set `cleanup_enabled = True` to run. This will not delete the Application Insights component or the hub.


In [ ]:
cleanup_enabled = False

if cleanup_enabled:
    apim_name = apimClientTool.apim_resource_name

    # 1) Delete the alert rules (scheduled query rules for logQuery mode, metric alerts for metric mode)
    for rule_name in ["ai-gateway-alertprobe-throttling-alert", "ai-gateway-alertprobe-quota-exceeded-alert"]:
        if alert_mode == "logQuery":
            del_cmd = f"az monitor scheduled-query delete -g {governance_hub_resource_group} -n {rule_name} --yes"
        else:
            del_cmd = f"az monitor metrics alert delete -g {governance_hub_resource_group} -n {rule_name}"
        utils.run(del_cmd, f"Deleted alert rule {rule_name}", "Could not delete alert rule (may not exist)")

    # 2) Delete the email action group
    utils.run(
        f"az monitor action-group delete -g {governance_hub_resource_group} -n ai-gateway-alertprobe-alerts-ag",
        "Deleted action group", "Could not delete action group (may not exist)",
    )

    # 3) Delete the APIM product (and its subscriptions) for the probe contract
    utils.run(
        f"az apim product delete --resource-group {governance_hub_resource_group} "
        f"--service-name {apim_name} --product-id {product_id} --delete-subscriptions true --yes",
        f"Deleted APIM product {product_id}", "Could not delete APIM product (may not exist)",
    )

    # 4) Remove generated parameter files
    for path in [alert_params_file, params_file, policy_path]:
        try:
            if os.path.exists(path):
                os.remove(path)
                utils.print_ok(f"Removed {path}")
        except Exception as e:
            utils.print_warning(f"Could not remove {path}: {e}")

    utils.print_ok("🧹 Cleanup complete.")
else:
    utils.print_info("Cleanup skipped (set cleanup_enabled = True to remove test resources).")
